In [2]:
import torch 
import numpy as np 

In [3]:
inputs = torch.tensor(
    [[0.43, 0.15, 0.89], # Your  (x^1)
     [0.55, 0.87, 0.66], # journey (x^2)
     [0.57, 0.85, 0.64], # starts (x^3)
     [0.22, 0.58, 0.33], # with (x^4)
     [0.77, 0.25, 0.10], # one (x^5)
     [0.05, 0.80, 0.55]] # step (x^6)
)
inputs.shape

torch.Size([6, 3])

In [4]:
query = inputs[1] # (x^2)
attn_scores_for_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs): 
    attn_scores_for_2[i] = torch.dot(x_i, query)
print(attn_scores_for_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [5]:
inputs[2]

tensor([0.5700, 0.8500, 0.6400])

In [6]:
vocabs = {
    0: "Your",
    1: "journey",
    2: "starts",
    3: "with",
    4: "one",
    5: "step"
}


In [7]:
print(vocabs[1])

journey


In [8]:
for idx, i in enumerate(attn_scores_for_2): 
    print(f"Attention between `{vocabs[1]}` and `{vocabs[idx]}`: {i}")

Attention between `journey` and `Your`: 0.9544000625610352
Attention between `journey` and `journey`: 1.4950001239776611
Attention between `journey` and `starts`: 1.4754000902175903
Attention between `journey` and `with`: 0.8434000015258789
Attention between `journey` and `one`: 0.7070000171661377
Attention between `journey` and `step`: 1.0865000486373901


In [9]:
attn_scores_for_2_tmp = attn_scores_for_2 / attn_scores_for_2.sum()
print(f"Attention weights: {attn_scores_for_2_tmp}")

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])


In [11]:
def softmax_naive(x): 
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_for_2)
print(f"Attention weights: {attn_weights_2_naive}")
print(f"Sum: {attn_weights_2_naive.sum()}")

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: 1.0


In [13]:
attn_weights_2 = torch.softmax(attn_scores_for_2, dim=0)
print(f"Attention weights: {attn_weights_2}")
print(f"Sum: {attn_weights_2.sum()}")

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: 1.0


In [14]:
query = inputs[1]
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs): 
    context_vec_2 += attn_weights_2[i] * x_i 
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


In [22]:
import time
start = time.time()

attn_scores = torch.empty(inputs.shape[0], inputs.shape[0])
for i, x_i in enumerate(inputs): 
    for j, x_j in enumerate(inputs): 
        attn_scores[i, j] = torch.dot(x_i, x_j)
end = time.time() 
print(attn_scores)
print(end-start)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
0.002321958541870117


In [23]:
start = time.time()

attn_scores = inputs @ inputs.T 
end = time.time() 

print(attn_scores)
print(end-start)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
0.001163482666015625


In [24]:
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [25]:
all_context_vecs = attn_weights @ inputs 
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])
